# Time-series forecasting

Predict future values from history: demand, traffic, sales, load.

Two ways to start, from Hello Model: https://sayhellomodel.com/models/forecasting/

**Your data:** the code uses example file names (such as `data.csv`). Upload your own file with the Files panel on the left, or change the name in the code. Run the cells from the top.

## Quick start

Pretrained forecasting model (Chronos / TimesFM) or Prophet: Zero-shot foundation models forecast without training; Prophet is simple and explainable.

In [ ]:
%pip install -q chronos-forecasting pandas torch

In [ ]:
# pip install chronos-forecasting pandas torch
import pandas as pd, torch
from chronos import BaseChronosPipeline

df = pd.read_csv("sales.csv", parse_dates=["date"])
pipe = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-small")
quantiles, mean = pipe.predict_quantiles(
    context=torch.tensor(df["sales"].values), prediction_length=28,
    quantile_levels=[0.1, 0.5, 0.9])
print(mean)

## Train your own model

StatsForecast (AutoETS/AutoARIMA) + LightGBM with lag features: Fast, reliable statistical models plus a tree model for extra signals.

In [ ]:
%pip install -q statsforecast pandas

In [ ]:
# pip install statsforecast pandas
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, SeasonalNaive

df = pd.read_csv("sales.csv", parse_dates=["date"])
df = df.rename(columns={"store_id": "unique_id", "date": "ds", "sales": "y"})

sf = StatsForecast(models=[AutoETS(season_length=7), SeasonalNaive(season_length=7)], freq="D")
cv = sf.cross_validation(df=df, h=28, n_windows=4)   # backtesting
print(cv.head())
forecast = sf.forecast(df=df, h=28)
forecast.to_csv("forecast.csv", index=False)